# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
meta_dict = json.loads(metadata.to_json())
print(f"{meta_dict['name']}: {meta_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record set @id values
record_sets = []
for rs in metadata.record_sets:
    print(f"RecordSet: {rs['@id']} ({rs['name']})")
    record_sets.append(rs['@id'])
    # List their fields/columns
    for field in rs['fields']:
        print(f" - Field @id: {field['@id']}, Name: {field['name']}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Load data from all available record sets
dataframes = {}

for record_set_id in record_sets:
    # List of record dicts
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set: {record_set_id}, shape={dataframes[record_set_id].shape}")
    else:
        print(f"No records found for record set: {record_set_id}")

# For demonstration, pick the first record set with data
example_rs = None
for rs in record_sets:
    if rs in dataframes:
        example_rs = rs
        break

if example_rs:
    print(f"Example DataFrame columns for record set {example_rs}:")
    print(dataframes[example_rs].columns.tolist())
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data by key attributes. This section includes operations like removing outliers, transforming data distributions, or grouping by attributes to prepare for analysis.

In [ ]:
# Choose a numeric field and group field by @id (from the record set columns)
import numpy as np

# Try to pick relevant field IDs for numeric and grouping
if example_rs:
    example_df = dataframes[example_rs]
    # Try to auto-detect a numeric column
    numeric_field = None
    for col in example_df.columns:
        if np.issubdtype(example_df[col].dtype, np.number):
            numeric_field = col
            break
    if not numeric_field:
        # Try to cast columns to numeric if possible
        for col in example_df.columns:
            try:
                converted = pd.to_numeric(example_df[col], errors='coerce')
                if converted.notna().any():
                    example_df[col] = converted
                    numeric_field = col
                    break
            except Exception:
                pass

    if numeric_field:
        print(f"Analyzing field: {numeric_field}")

        # Set threshold to mean for demo purposes
        threshold = example_df[numeric_field].mean()
        filtered_df = example_df[example_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (threshold is mean):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std(ddof=0)
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to select a group field (categorical)
        group_field = None
        for col in example_df.columns:
            if example_df[col].dtype == object and col != numeric_field:
                group_field = col
                break
        if group_field:
            print(f"Grouping by: {group_field}")
            grouped_df = (
                filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            )
            print("Grouped mean:")
            display(grouped_df.head())
    else:
        print("No numeric field found to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_rs and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[example_rs][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {example_rs}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping field exists, plot boxplot
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[example_rs])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded dataset metadata and records using the Croissant schema and `mlcroissant`.
- Inspected the schema structure, including record sets and field `@id`s.
- Loaded data into DataFrames, filtered and visualized a numeric attribute, and explored categorical groupings.

You may further tailor processing or visualizations using precise field `@id`s and domain knowledge from the dataset documentation.